## 1. 데이터 살펴 보기
- 이진 분류는 0,1 출력 ( 시그모이드 사용)
- 다중 클래스 분류는 softmax 사용하여 확률 분포로 나타낸다.
(이런 경우 일반적으로 샘플은 하나의 클래스에 속한다)
- 단일 레이블 분류 : 샘플이 하나의 클래스에만 속하는 경우
- 다중 레이블 분류 : 샘플이 여러 개의 클래스에 속하는 경우

In [3]:
from sklearn.datasets import load_iris

dataset = load_iris()
dataset.keys()


dict_keys(['data', 'target', 'frame', 'target_names', 'DESCR', 'feature_names', 'filename', 'data_module'])

In [4]:
print(dataset.DESCR)

.. _iris_dataset:

Iris plants dataset
--------------------

**Data Set Characteristics:**

:Number of Instances: 150 (50 in each of three classes)
:Number of Attributes: 4 numeric, predictive attributes and the class
:Attribute Information:
    - sepal length in cm
    - sepal width in cm
    - petal length in cm
    - petal width in cm
    - class:
            - Iris-Setosa
            - Iris-Versicolour
            - Iris-Virginica

:Summary Statistics:

============== ==== ==== ======= ===== ====================
                Min  Max   Mean    SD   Class Correlation
============== ==== ==== ======= ===== ====================
sepal length:   4.3  7.9   5.84   0.83    0.7826
sepal width:    2.0  4.4   3.05   0.43   -0.4194
petal length:   1.0  6.9   3.76   1.76    0.9490  (high!)
petal width:    0.1  2.5   1.20   0.76    0.9565  (high!)
============== ==== ==== ======= ===== ====================

:Missing Attribute Values: None
:Class Distribution: 33.3% for each of 3 classes.
:Cr

- 붓꽃 데이터셋은 총 150개의 샘플을 갖고 있으며 꽃받침 길이, 너비, 꽃잎 깊이, 너비와 4개의 특징을 제공한다.
- 이러한 특징을 이용해서 3개의 클래스로 구분하는 모델을 학습해보겠습니다.


In [5]:
import pandas as pd

data = pd.DataFrame(dataset.data, columns = dataset.feature_names)
data['target'] = dataset.target

print(data.head())

   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  \
0                5.1               3.5                1.4               0.2   
1                4.9               3.0                1.4               0.2   
2                4.7               3.2                1.3               0.2   
3                4.6               3.1                1.5               0.2   
4                5.0               3.6                1.4               0.2   

   target  
0       0  
1       0  
2       0  
3       0  
4       0  


## 2. 파이토치 데이터 유틸 사용하기
- 데이터를 학습에 사용하기 전 특별한 전처리를 한다거나 여러 파일의 데이터를 가져와야 할 수도 있습니다.
- 이를 하나의 클래스로 묶어서 편리하게 사용하기 위해 파이토치의 데이터 유틸에서는 
- Dataset과 Dataloader 클래스를 제공하고 있습니다.

In [6]:
# 랜덤 셔플
# 일반적으로 학습 시 데이터 순서를 랜덤하게 섞어주는 작업을 하는데 이를 랜덤 셔플이라고 한다.
# 랜덤 셔플은 모델이 데이터 순서를 학습하는 편향을 막기 위해 사용한다.

import torch
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split

# __ __ 붙은 함수는 매직 메서드 
# 파이썬 내부적으로 특정 문법이나 객체 동작을 처리하기 위해 이미 약속해둔 특별한 함수들입니다. 
# 개발자가 직접 dataset.__getitem__(0)처럼 호출하기보다는 파이썬 문법을 사용할 때 자동으로 실행됩니다.


## __가 앞에 만 붙은 함수는 프라이빗 메서드 

class IrisDataset(Dataset):
    def __init__(self, train = True):
        dataset = load_iris()
        X_train,X_test, y_train, y_test = train_test_split(
            dataset.data, dataset.target, test_size=0.3, random_state =827
        )
        
        if train :
            self.data = torch.FloatTensor(X_train)
            self.target = torch.LongTensor(y_train)
        else :
            self.data = torch.FloatTensor(X_test)
            self.target = torch.LongTensor(y_test)

            
    def __getitem__(self,i):
        return self.data[i],self.target[i]

    def __len__(self):
        return len(self.data)
            

## 3. 모델 구현 및 학습


In [12]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# 하이퍼파라미터
batch_size = 64
learning_rate = 1e-3
epochs = 2000

# 모델
model = nn.Sequential(
    nn.Linear(4,128),
    nn.ReLU(),
    nn.Linear(128,64),
    nn.ReLU(),
    nn.Linear(64,3)
)

# 데이터셋, 데이터 로더
train_dataset = IrisDataset(train= True)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle= True)

# optimizer
optimizer = torch.optim.Adam(params = model.parameters(),lr=learning_rate)

# loss 
criterion = nn.CrossEntropyLoss()

for epoch in range(epochs):
    #배치
    for data,target in train_dataloader:
        optimizer.zero_grad()
        
        pred = model(data)
        
        loss = criterion(pred, target)
        loss.backward()
        optimizer.step()
        
        #학습 기록
        if epoch % 100 == 99:
            print("epoch", epoch +1, "loss", loss.item())

epoch 100 loss 0.0979623794555664
epoch 100 loss 0.10455994307994843
epoch 200 loss 0.08394347876310349
epoch 200 loss 0.06141044944524765
epoch 300 loss 0.07162539660930634
epoch 300 loss 0.055829379707574844
epoch 400 loss 0.07086259126663208
epoch 400 loss 0.04764832928776741
epoch 500 loss 0.06460997462272644
epoch 500 loss 0.06982558220624924
epoch 600 loss 0.0372837670147419
epoch 600 loss 0.10193395614624023
epoch 700 loss 0.08384130895137787
epoch 700 loss 0.01680140197277069
epoch 800 loss 0.056035272777080536
epoch 800 loss 0.050562161952257156
epoch 900 loss 0.05685523897409439
epoch 900 loss 0.041181184351444244
epoch 1000 loss 0.03948650881648064
epoch 1000 loss 0.07477284222841263
epoch 1100 loss 0.07096271961927414
epoch 1100 loss 0.015190179459750652
epoch 1200 loss 0.07196707278490067
epoch 1200 loss 0.01760665886104107
epoch 1300 loss 0.03263663873076439
epoch 1300 loss 0.0733850970864296
epoch 1400 loss 0.02994849905371666
epoch 1400 loss 0.08483242988586426
epoch 15

## 4. 모델 성능 평가
- 분류 문제는 예측값이 타겟값과 일치하는 정확도를 성능 평가 지표로 사용합니다.


In [13]:
test_dataset = IrisDataset(train = False)
test_dataloader = DataLoader(test_dataset, batch_size= batch_size,shuffle = False)

num_correct = 0

with torch.no_grad():
    for data, target in test_dataloader:
        output = model(data)
        pred = torch.max(output,1)[1]
        
        corr = pred.eq(target).sum().item()
        num_correct += corr
        
        print("Accuracy:", (num_correct/len(test_dataset.data))*100, "%")

Accuracy: 93.33333333333333 %
